# 01 · Coleta e chunking

Baixa as fontes de `base/fontes.md` (exceto o GTFS e as fontes descartadas), extrai o texto limpo, divide em trechos e grava `base/trechos.jsonl`.

- Leis: um artigo = um trecho (mais de um se passar de 380 palavras). Da LBI ficam só os arts. 46 a 52.
- `metro_elev`: tabela do PDF (pdfplumber), uma frase por local; um trecho por estação e linha (v2 da etapa 3; a v1 agrupava por linha do Metrô).
- Demais fontes: trechos de 250 a 380 palavras, 40 palavras de sobreposição, cortados só em fim de frase.
- Se o download falhar, usa o arquivo salvo manualmente em `base/brutos/<id>.html` e avisa.
- Os originais ficam versionados em `base/brutos/`.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ / 'src'))

from sonar_ia import base

todas = base.ler_fontes()
fontes = [f for f in todas if f['id'] != 'gtfs_sptrans' and not f['descartada']]
print('Descartadas:', [f['id'] + ' (' + f['responde_a'] + ')' for f in todas if f['descartada']])
print(len(fontes), 'fontes:', [f['id'] for f in fontes])

Descartadas: ['metro_guia (descartada — URL 404 e sem cópia no Wayback em 16/11/2026)']
9 fontes: ['lbi', 'lei_caoguia', 'dec_caoguia', 'smped_guia', 'sptrans_manual', 'sptrans_pcd', 'metro_elev', 'metro_pref', 'cptm_acess']


## 1. Download (original salvo em `base/brutos/<id>.<ext>`)

In [2]:
brutos = {}
for f in fontes:
    b = base.obter_bruto(f)
    brutos[f['id']] = b
    status = 'OK' if b['aviso'] is None else 'ATENÇÃO'
    print(f"[{status}] {f['id']:<15} -> {b['caminho'].name if b['caminho'] else '—'}")
    if b['aviso']:
        print('         ', b['aviso'])

precisaram_manual = [i for i, b in brutos.items() if b['aviso']]
print('\nFontes que precisaram de arquivo manual ou ficaram sem conteúdo:', precisaram_manual or 'nenhuma')

[OK] lbi             -> lbi.html
[OK] lei_caoguia     -> lei_caoguia.html
[OK] dec_caoguia     -> dec_caoguia.html
[OK] smped_guia      -> smped_guia.html
[OK] sptrans_manual  -> sptrans_manual.pdf
[OK] sptrans_pcd     -> sptrans_pcd.html
[OK] metro_elev      -> metro_elev.pdf
[OK] metro_pref      -> metro_pref.html
[OK] cptm_acess      -> cptm_acess.html

Fontes que precisaram de arquivo manual ou ficaram sem conteúdo: nenhuma


## 2. Texto limpo e chunking

In [3]:
textos, trechos = {}, []
for f in fontes:
    b = brutos[f['id']]
    if b['caminho'] is None:
        print(f"[SEM TEXTO] {f['id']}")
        continue
    textos[f['id']] = base.extrair_texto(f['id'], b['caminho'])
    novos = base.gerar_trechos(f, textos[f['id']], b['url'], b['formato'], b['data_coleta'])
    palavras = [len(t['texto'].split()) for t in novos]
    faixa = f'{min(palavras)}-{max(palavras)}' if palavras else '—'
    print(f"{f['id']:<15} texto: {len(textos[f['id']].split()):>5} palavras | trechos: {len(novos):>3} | palavras por trecho: {faixa}")
    trechos += novos

lbi             texto: 16686 palavras | trechos:   7 | palavras por trecho: 29-243
lei_caoguia     texto:   267 palavras | trechos:   4 | palavras por trecho: 12-99
dec_caoguia     texto:  1595 palavras | trechos:   9 | palavras por trecho: 13-314
smped_guia      texto:  1329 palavras | trechos:   4 | palavras por trecho: 350-380


sptrans_manual  texto:  4608 palavras | trechos:  15 | palavras por trecho: 334-380
sptrans_pcd     texto:    29 palavras | trechos:   1 | palavras por trecho: 29-29


metro_elev      texto:  2250 palavras | trechos:  66 | palavras por trecho: 14-84
metro_pref      texto:   238 palavras | trechos:   1 | palavras por trecho: 238-238
cptm_acess      texto:   241 palavras | trechos:   1 | palavras por trecho: 241-241


## 3. Gravar `base/trechos.jsonl` e atualizar `base/fontes.md`

In [4]:
from collections import Counter

base.salvar_trechos(trechos)
contagem = Counter(t['fonte'] for t in trechos)
base.atualizar_contagem_fontes({f['id']: contagem.get(f['id'], 0) for f in fontes})

print('Trechos por fonte:')
for f in fontes:
    print(f"  {f['id']:<15} {contagem.get(f['id'], 0):>3}")
print('  TOTAL'.ljust(18), len(trechos))

Trechos por fonte:
  lbi               7
  lei_caoguia       4
  dec_caoguia       9
  smped_guia        4
  sptrans_manual   15
  sptrans_pcd       1
  metro_elev       66
  metro_pref        1
  cptm_acess        1
  TOTAL            108


## 4. Conferência: `sptrans_pcd` (texto completo) e `metro_elev` (tabela × frase)

In [5]:
print(textos['sptrans_pcd'])
print('\n(palavras:', len(textos['sptrans_pcd'].split()), ')')

Informamos que o Bilhete Único Especial da Pessoa com Deficiência deverá ser solicitado pelo
Site de Atendimento
,
mediante cadastro e envio de dados pessoais do(a) usuário(a) em questão.

(palavras: 29 )


In [6]:
conteudo = (base.DIR_BRUTOS / 'metro_elev.pdf').read_bytes()
entradas = base.tabela_metro_elev(conteudo)
from collections import Counter
print('locais lidos da tabela:', len(entradas))
for linha in dict.fromkeys(e['linha'] for e in entradas):
    estacoes = list(dict.fromkeys(e['estacao'] for e in entradas if e['linha'] == linha))
    print(f'  {linha:<18} {len(estacoes):>2} estações, {sum(e["linha"] == linha for e in entradas):>2} locais')
assert all(e['acesso_estacao'] and e['acesso_plataforma'] for e in entradas), 'célula de acesso vazia'

amostra = [('TUCURUVI', 'Linha 1-AZUL'), ('SANTA CRUZ', 'Linha 1-AZUL'), ('PALMEIRAS-BARRA FUNDA', 'Linha 3-VERMELHA'),
          ('PARAÍSO', 'Linha 2-VERDE'), ('JABAQUARA', 'Linha 1-AZUL')]
for nome, linha in amostra:
    e = [x for x in entradas if x['estacao'] == nome and x['linha'] == linha][-1 if nome == 'JABAQUARA' else 0]
    print('\n' + '-' * 100)
    print('PDF   :', nome, '|', base.original_da_entrada(conteudo, e))
    print('FRASE :', base.frase_metro_elev(e))

locais lidos da tabela: 119
  Linha 1-AZUL       23 estações, 35 locais
  Linha 2-VERDE      14 estações, 24 locais
  Linha 3-VERMELHA   18 estações, 38 locais
  Linha 15-PRATA     11 estações, 22 locais

----------------------------------------------------------------------------------------------------
PDF   : TUCURUVI | TUCURUVI Av. Dr. Antônio Maria de | Rampa Elevador | Laet
FRASE : Estação Tucuruvi (Linha 1-Azul), local Av. Dr. Antônio Maria de Laet: acesso à estação: Rampa; acesso à plataforma: Elevador.

----------------------------------------------------------------------------------------------------
PDF   : SANTA CRUZ | SANTA CRUZ Shopping Metrô Santa | Elevador¹ Elevador | Cruz
FRASE : Estação Santa Cruz (Linha 1-Azul), local Shopping Metrô Santa Cruz: acesso à estação: Elevador (Equipamento sob responsabilidade do Shopping Metrô Santa Cruz); acesso à plataforma: Elevador.

----------------------------------------------------------------------------------------------------

PDF   : PALMEIRAS-BARRA FUNDA | PALMEIRAS- Terminal Turístico
FRASE : Estação Palmeiras-Barra Funda (Linha 3-Vermelha), local Terminal Turístico: acesso à estação: Rampa; acesso à plataforma: Elevador.

----------------------------------------------------------------------------------------------------
PDF   : PARAÍSO | PARAÍSO Rua Vergueiro próx. À | Catedral Ortodoxo | Elevador Elevador
FRASE : Estação Paraíso (Linha 2-Verde), local Rua Vergueiro próx. À Catedral Ortodoxo: acesso à estação: Elevador; acesso à plataforma: Elevador.

----------------------------------------------------------------------------------------------------
PDF   : JABAQUARA | 3 | Terminal EMTU Elevador
FRASE : Estação Jabaquara (Linha 1-Azul), local Terminal EMTU: acesso à estação: Elevador (Equipamento sob responsabilidade da EMTU); acesso à plataforma: Elevador.


## 5. Validação do `trechos.jsonl` e exemplos

In [7]:
import json

carregados = base.carregar_trechos()          # valida os 7 metadados em todas as linhas
assert len(carregados) == len(trechos)
assert len({t['id'] for t in carregados}) == len(carregados), 'ids repetidos'
print('trechos.jsonl OK:', len(carregados), 'trechos, ids únicos, metadados completos')

for id_ in ('lbi_001', 'metro_elev_l1-azul_paraiso'):
    t = next(t for t in carregados if t['id'] == id_)
    print('\n' + json.dumps({**t, 'texto': t['texto'][:500] + ('…' if len(t['texto']) > 500 else '')}, ensure_ascii=False, indent=2))

trechos.jsonl OK: 108 trechos, ids únicos, metadados completos

{
  "id": "lbi_001",
  "fonte": "lbi",
  "url": "https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm",
  "formato": "HTML",
  "licenca": "Texto legal, sem proteção autoral (Lei 9.610/98, art. 8º, IV)",
  "data_coleta": "2026-11-16",
  "texto": "Art. 46. O direito ao transporte e à mobilidade da pessoa com deficiência ou com mobilidade reduzida será assegurado em igualdade de oportunidades com as demais pessoas, por meio de identificação e de eliminação de todos os obstáculos e barreiras ao seu acesso.\n§ 1º Para fins de acessibilidade aos serviços de transporte coletivo terrestre, aquaviário e aéreo, em todas as jurisdições, consideram-se como integrantes desses serviços os veículos, os terminais, as estações, os pontos de parada, o sis…"
}

{
  "id": "metro_elev_l1-azul_paraiso",
  "fonte": "metro_elev",
  "url": "https://www.metro.sp.gov.br/wp-content/uploads/2023/05/elevadores-metro-formas-acesso.pdf